# ESA CCI Biomass V5.01 — Access and Visualization (Token-secured)

Authors: Rajat Shinde (UAH), Alex Mandel (Development Seed), Sheyenne Kirkland (UAH), Harshini Girish (UAH), Jamison French (Development Seed), Henry Rodman (Development Seed), Chuck Daniels (Development Seed), Zac Deziel (Development Seed), Brian Freitag (NASA)

Date: August 29, 2025

Description: This notebook documents how to access and visualize the EarthCARE dataset hosted on the ESA MAAP server. It is an example illustrating data access from ESA server based on ESA MAAP Token using the NASA MAAP Authorization.



## What you will do
1. Understand the product and file organization.  
2. Obtain an ESA access token.  
3. Access using the token.
4. Check the internal format compatibility with advanced data access methods for HDF5
5. Visualize in Python.



## Run This Notebook

To access and run this tutorial within MAAP’s Algorithm Development Environment (ADE), please refer to the [Getting started with the MAAP](#) section of our documentation.

**Disclaimer**: It is highly recommended to run this tutorial within MAAP’s ADE, which already includes packages specific to MAAP, such as maap-py. Running the tutorial outside of the MAAP ADE may lead to errors.

**Prerequisites**  
- An active ESA MAAP portal account with access initialized.  
- OAuth2 client credentials for your ESA realm.  
- Python packages: `requests`, `rasterio`, `numpy`, `matplotlib` (optional: `pystac-client`, `stackstac`).  


## Importing and Installing Packages

In [96]:
# Install if needed. Comment out if already available.
# !mamba install -y -c conda-forge rasterio xarray matplotlib fsspec requests
# !pip install pystac-client stackstac

import os
import stat
import getpass
import pathlib
from tqdm import tqdm

import requests
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt

from pystac_client import Client

import fsspec
import xarray as xr
import pandas as pd 
import geopandas as gpd
from IPython.display import Image, display



## Getting the Token from the ESA MAAP portal

This explains how to retrieve a short-lived access token from the ESA MAAP portal using your browser and NASA EDL login.

Open the token page in your browser:
**https://portal.maap.eo.esa.int/ini/services/auth/token/**

**Steps**
1. Navigate to the URL above.  
2. Choose **NASA Earthdata Login (EDL)** when prompted and authorize access.  
3. After successful authorization you will see a **token page** showing your short‑lived access token string.

**The below screenshots illustrate the process for each steps.**
- **Portal entry page**:  
  ![Portal](./images/esa_maap.jpg)
- **NASA EDL authorization screen**:  
  ![NASA EDL](./images/page2.jpg)
- **Token page after authorization**:  
  ![Token Page](./images/page3.jpg)

**Copy the token value** from the token page for use in the next cell.

**Notes**
- Tokens are short‑lived. If you see **401 Unauthorized** later, refresh the token using the same URL and update the value in the notebook.
- Treat tokens as secrets. Do not commit them to version control or share publicly.

Now we paste the token we got from the portal and save it to a file so it can be used later in the notebook.

In [73]:
TOKEN_FILE = pathlib.Path.home() / ".config" / "esa_maap" / "tokens"
TOKEN_FILE.parent.mkdir(parents=True, exist_ok=True)

if not TOKEN_FILE.exists():
    tok = getpass.getpass("Paste ESA portal token (hidden): ").strip()
    if not tok:
        raise ValueError("Empty token.")
    TOKEN_FILE.write_text(tok, encoding="utf-8")
    
    TOKEN_FILE.chmod(stat.S_IRUSR | stat.S_IWUSR)


st = TOKEN_FILE.stat()
if (st.st_mode & 0o777) != 0o600:
    raise PermissionError(f"{TOKEN_FILE} must have mode 600. Fix with: chmod 600 {TOKEN_FILE}")

ESA_TOKEN = TOKEN_FILE.read_text(encoding="utf-8").strip()
print("Token loaded from file:", TOKEN_FILE)


Paste ESA portal token (hidden):  ········


Token loaded from file: /projects/.config/esa_maap/tokens


## Discover tiles via ESA STAC

Now, we will query the ESA STAC API for CCIBiomassV5.01 over the given bounding box and time range, and then select the COG data assets URLs for this AOI and time range. We handle pagination, prefer assets labeled as COG data, and record the AOI window for efficient partial reads.



In [90]:
STAC_URL = "https://catalog.maap.eo.esa.int/catalogue/"
COLLECTION = "EarthCAREL2Validated_MAAP"
BBOX = [10.0, 0.0, 10.6, 0.6]               
DT = "2010-01-01/2010-12-31" #find a valid date time filter
cql2_filter = {"op": "=",
               "args": [
                {
                  "property": "productType"
                },
                "MSI_COP_2A"
                ]
            }

cql2_text = " productType='MSI_COP_2A' and frame='E' "
params = {
    "method": "GET", # Need this OR filter_lang cql2-text
    "collections": COLLECTION, 
    "bbox": BBOX, 
    "max_items":10,
    "filter": cql2_text,
    "filter_lang": "cql2-text",
}


In [91]:
api   = Client.open(STAC_URL)
search = api.search(**params)


In [93]:
#TODO: Improve this by normalizing the json, bring the asset hrefs to the top level
#gdf = gpd.GeoDataFrame.from_features(search.item_collection_as_dict())

In [94]:
# We just need 1 item for testing
items = list(search.get_items())
items[1]

<Item id=ECA_EXAB_MSI_COP_2A_20250705T131451Z_20250705T162240Z_06264E>

In [69]:
def get_hrefs(item):
    for asset in item.assets.values():
        if ("application/x-hdf5" in asset.media_type.lower()) or asset.href.lower().endswith(".h5"):
            return(asset.href)

In [70]:
granule_urls = [get_hrefs(item) for item in items]

In [72]:
# Use the first item from our STAC query as an example tile
EXAMPLE_URL = granule_urls[1]

print("Using example tile from STAC query:", EXAMPLE_URL)


Using example tile from STAC query: https://catalog.maap.eo.esa.int/data/earthcare-pdgs-01/EarthCARE/MSI_COP_2A/AB/2025/05/09/ECA_EXAB_MSI_COP_2A_20250509T144033Z_20250509T161638Z_05378E/ECA_EXAB_MSI_COP_2A_20250509T144033Z_20250509T161638Z_05378E/ECA_EXAB_MSI_COP_2A_20250509T144033Z_20250509T161638Z_05378E.h5


## Get a Sample to check file layout

Using an ESA token we'll download a sample single granule and explore the data structure of the file.

In [74]:
headers = {"Authorization": f"Bearer {ESA_TOKEN}"}
r = requests.get(EXAMPLE_URL, headers=headers, stream=True)

print("HTTP status:", r.status_code)
if r.status_code == 403:
    print(
        "403 Forbidden: Your account may need initialization for this collection.\n"
        "Follow any initialization link provided by the server, refresh the token, and retry."
    )
elif r.status_code == 401:
    print("401 Unauthorized: Token expired or invalid. Get a new token at the portal URL and update ESA_TOKEN.")
elif r.status_code != 200:
    print("Unexpected status. Check URL, permissions, or try another tile/year.")
else:
    print("Access OK.")


HTTP status: 200
Access OK.


In [81]:
def download_file_with_bearer_token(url, token, file_path, disable_bar=False):
  """
  Downloads a file from a given URL using a Bearer token.
  """

  try:
    headers = {"Authorization": f"Bearer {token}"}
    response = requests.get(url, headers=headers, stream=True)
    response.raise_for_status()  # Raise an exception for bad status codes
    file_size = int(response.headers.get('content-length', 0))

    chunk_size = 8 * 1024 * 1024 # Byes - 1MiB
    with open(file_path, "wb") as f, tqdm(
        desc=file_path,
        total=file_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
        disable=disable_bar,
      ) as bar:
      for chunk in response.iter_content(chunk_size=chunk_size):
        read_size=f.write(chunk)
        bar.update(read_size)

    if (disable_bar): 
      print(f"File downloaded successfully to {file_path}")

  except requests.exceptions.RequestException as e:
    print(f"Error downloading file: {e}")

In [84]:
local_file = f"/tmp/{os.path.basename(EXAMPLE_URL)}"
download_file_with_bearer_token(EXAMPLE_URL, ESA_TOKEN, local_file)

/tmp/ECA_EXAB_MSI_COP_2A_20250509T144033Z_20250509T161638Z_05378E.h5: 100%|██████████| 130M/130M [00:14<00:00, 9.42MiB/s] 


## Explore the Data

Open the file with various tools to explore the chunking and page layout.

In [85]:
!h5stat -S {local_file}

Filename: /tmp/ECA_EXAB_MSI_COP_2A_20250509T144033Z_20250509T161638Z_05378E.h5
File space management strategy: H5F_FSPACE_STRATEGY_FSM_AGGR
File space page size: 4096 bytes
Summary of file space information:
  File metadata: 69985 bytes
  Raw data: 136199589 bytes
  Amount/Percent of tracked free space: 0 bytes/0.0%
  Unaccounted space: 32111 bytes
Total space: 136301685 bytes


In [89]:
!h5dump -pH {local_file} | grep cloud_water -A 10

      DATASET "cloud_water_path" {
         DATATYPE  H5T_IEEE_F32LE
         DATASPACE  SIMPLE { ( 11904, 384 ) / ( 11904, 384 ) }
         STORAGE_LAYOUT {
            CHUNKED ( 3968, 128 )
            SIZE 6264481 (2.919:1 COMPRESSION)
         }
         FILTERS {
            PREPROCESSING SHUFFLE
            COMPRESSION DEFLATE { LEVEL 3 }
         }
--
      DATASET "cloud_water_path_error" {
         DATATYPE  H5T_IEEE_F32LE
         DATASPACE  SIMPLE { ( 11904, 384 ) / ( 11904, 384 ) }
         STORAGE_LAYOUT {
            CHUNKED ( 3968, 128 )
            SIZE 6250575 (2.925:1 COMPRESSION)
         }
         FILTERS {
            PREPROCESSING SHUFFLE
            COMPRESSION DEFLATE { LEVEL 3 }
         }


## Test Accessing the Data over HTTPS without downloading 1st

Working toward cloud native access, we're going to test opening the file with Xarray over the network.`

In [98]:
io_params = {
    "fsspec_params": {
        "cache_type": "blockcache",
        "block_size": 8 * 1024 * 1024
    },
    "h5py_params": {
        "driver_kwds": {
            #"page_buf_size": 16 * 1024 * 1024, #doesn't work because File space management strategy is not Page based
            "rdcc_nbytes": 4 * 1024 * 1024
        }
    }
}

fs = fsspec.filesystem(
    "https", 
    headers={"Authorization": f"Bearer {ESA_TOKEN}"}, 
    **io_params["fsspec_params"]  
)



In [102]:
%%time
ds = xr.open_dataset(
    fs.open(EXAMPLE_URL, "rb", **io_params["fsspec_params"]),
    engine="h5netcdf",
    group="ScienceData",
    chunks="auto",
    **io_params["h5py_params"]
)
ds

CPU times: user 129 ms, sys: 32.9 ms, total: 162 ms
Wall time: 4.83 s


<xarray.Dataset> Size: 297MB
Dimensions:                        (along_track: 11904, across_track: 384)
Dimensions without coordinates: along_track, across_track
Data variables: (12/18)
    time                           (along_track) datetime64[ns] 95kB dask.array<chunksize=(11904,), meta=np.ndarray>
    latitude                       (along_track, across_track) float64 37MB dask.array<chunksize=(11904, 384), meta=np.ndarray>
    longitude                      (along_track, across_track) float64 37MB dask.array<chunksize=(11904, 384), meta=np.ndarray>
    geoid_offset                   (along_track) float32 48kB dask.array<chunksize=(11904,), meta=np.ndarray>
    missing_lines_before_flag      (along_track) int8 12kB dask.array<chunksize=(11904,), meta=np.ndarray>
    quality_status                 (along_track, across_track) int8 5MB dask.array<chunksize=(11904, 384), meta=np.ndarray>
    ...                             ...
    cloud_top_pressure             (along_track, across_track) float32 18MB dask.array<chunksize=(11904, 384), meta=np.ndarray>
    cloud_top_temperature          (along_track, across_track) float32 18MB dask.array<chunksize=(11904, 384), meta=np.ndarray>
    cloud_top_height               (along_track, across_track) float32 18MB dask.array<chunksize=(11904, 384), meta=np.ndarray>
    cloud_top_pressure_error       (along_track, across_track) float32 18MB dask.array<chunksize=(11904, 384), meta=np.ndarray>
    cloud_top_temperature_error    (along_track, across_track) float32 18MB dask.array<chunksize=(11904, 384), meta=np.ndarray>
    cloud_top_height_error         (along_track, across_track) float32 18MB dask.array<chunksize=(11904, 384), meta=np.ndarray>